### This notebook downloads and preprocesses raw data
For downloading raw data, it queries APIs.  
For preprocessing, it currently clips to an NC mask (data/processed/nc_boundary.gpkg)


In [ ]:
import geopandas as gpd
import earthaccess
from collections import defaultdict
import geopandas as gpd
import rioxarray
from rioxarray.merge import merge_arrays
from peatfire import data_path
from pathlib import Path
import xarray as xr

ModuleNotFoundError: No module named 'h5netcdf'

nc boundary download

In [ ]:
url = 'https://www2.census.gov/geo/tiger/GENZ2018/shp/cb_2018_us_state_500k.zip'
states = gpd.read_file(url)            # geopandas reads the zip directly
nc = states[states['NAME'] == 'North Carolina']
nc.to_file('nc_boundary.gpkg', driver='GPKG')   # GeoPackage > shapefile

earthaccess bulk download of MCD64A1

In [ ]:
earthaccess.login()  # uses Earthdata credentials / .netrc

results = earthaccess.search_data(
    short_name='MCD64A1',
    version='061',
    temporal=('2017-01-01', '2018-05-01'),
    bounding_box=(-84.5, 33.7, -75.4, 36.6),  # NC bbox; returns h11v05 + h12v05
)
earthaccess.download(results, '../data/raw/fire/MCD64A1_061/')

/Users/jinjiang-macair/anaconda3/envs/peat_fire_stanback/lib/python3.11/site-packages/earthaccess/results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()
/Users/jinjiang-macair/anaconda3/envs/peat_fire_stanback/lib/python3.11/site-packages/earthaccess/store.py:832: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)
QUEUEING TASKS | : 100%|██████████| 34/34 [00:00<00:00, 6204.32it/s]
PROCESSING TASKS | : 100%|██████████| 34/34 [00:15<00:00,  2.19it/s]
COLLECTING RESULTS | : 100%|██████████| 34/34 [00:00<00:00, 268057.02it/s]


[PosixPath('../data/raw/fire/MCD64AQ_061/MCD64A1.A2017001.h10v05.061.2022017151205.hdf'),
 PosixPath('../data/raw/fire/MCD64AQ_061/MCD64A1.A2017001.h11v05.061.2022017150916.hdf'),
 PosixPath('../data/raw/fire/MCD64AQ_061/MCD64A1.A2017032.h11v05.061.2021312112736.hdf'),
 PosixPath('../data/raw/fire/MCD64AQ_061/MCD64A1.A2017032.h10v05.061.2021312112619.hdf'),
 PosixPath('../data/raw/fire/MCD64AQ_061/MCD64A1.A2017060.h10v05.061.2021312114829.hdf'),
 PosixPath('../data/raw/fire/MCD64AQ_061/MCD64A1.A2017060.h11v05.061.2021312114918.hdf'),
 PosixPath('../data/raw/fire/MCD64AQ_061/MCD64A1.A2017091.h11v05.061.2021312121124.hdf'),
 PosixPath('../data/raw/fire/MCD64AQ_061/MCD64A1.A2017091.h10v05.061.2021312121039.hdf'),
 PosixPath('../data/raw/fire/MCD64AQ_061/MCD64A1.A2017121.h10v05.061.2021312123125.hdf'),
 PosixPath('../data/raw/fire/MCD64AQ_061/MCD64A1.A2017121.h11v05.061.2021312123226.hdf'),
 PosixPath('../data/raw/fire/MCD64AQ_061/MCD64A1.A2017152.h11v05.061.2021312125324.hdf'),
 PosixPath

clip MCD64A1 to NC mask

In [ ]:
def sds(hdf):
    """Return the GDAL subdataset string for the Burn Date layer of an MCD64A1 file."""
    with rasterio.open(str(hdf)) as src:
        for name in src.subdatasets:
            if name.rstrip().endswith("Burn Date"):
                return name
    raise ValueError(f"No 'Burn Date' subdataset found in {hdf.name}")

# 1. NC Boundary
nc = gpd.read_file(data_path('processed', 'nc_boundary.gpkg'))

# 2. group the two tiles by acquisition date (filename token A2017001, A2017032, ...)
raw = data_path('raw', 'fire', 'MCD64A1_061')
by_date = defaultdict(list)
for f in sorted(raw.glob("MCD64A1.*.hdf")):
    by_date[f.name.split(".")[1]].append(f)

# 3. mosaic -> clip -> save, per date
out = data_path('processed', 'fire', 'MCD64A1_061')
out.mkdir(parents=True, exist_ok=True)

for date, files in by_date.items():
    tiles = [rioxarray.open_rasterio(sds(f), masked=True) for f in files]
    mosaic = merge_arrays(tiles) # stitch h10v05 + h11v05
    nc_sin = nc.to_crs(mosaic.rio.crs) # reproject NC -> sinusoidal
    clip = mosaic.rio.clip(nc_sin.geometry, nc_sin.crs, drop=True)
    clip.rio.to_raster(out / f"MCD64A1_{date}_nc.tif")
    

/Users/jinjiang-macair/anaconda3/envs/peat_fire_stanback/lib/python3.11/site-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)
/Users/jinjiang-macair/anaconda3/envs/peat_fire_stanback/lib/python3.11/site-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)
/Users/jinjiang-macair/anaconda3/envs/peat_fire_stanback/lib/python3.11/site-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)
/Users/jinjiang-macair/anaconda3/envs/peat_fire_stanback/lib/python3.11/site-packages/rasterio/__init__.py:356: NotGeoref